# AI in Healthcare - Lab Experiment 1

**Name:** Vanshika Jain
**Course:** CSET343 - AI in Healthcare
**Objective:** Read tabular, textual, image and signal data in different formats and explore them.

## 1. Setup
Installing the libraries I need for this lab.

In [ ]:
!pip install -q pandas scikit-learn wfdb matplotlib seaborn numpy pillow

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

## 2. Tabular Data - Heart Disease Dataset

Using the Cleveland Heart Disease dataset from UCI. It has 14 columns after cleaning, target column tells if the patient has heart disease or not.

In [ ]:
cols = ["age","sex","cp","trestbps","chol","fbs","restecg","thalach",
        "exang","oldpeak","slope","ca","thal","target"]

url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"
df = pd.read_csv(url, names=cols, na_values="?")

df.shape

In [ ]:
df.head()

In [ ]:
# checking for null values
df.isnull().sum()

In [ ]:
# dropping rows with missing values, only a few so should be fine
df = df.dropna()

# target has values 0-4, converting to binary (0 = no disease, 1 = disease)
df["target"] = df["target"].apply(lambda x: 1 if x > 0 else 0)
df["target"].value_counts()

In [ ]:
# quick look at the data
plt.figure(figsize=(6,4))
sns.histplot(df["age"], kde=True)
plt.title("Age distribution")
plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(df.corr(), cmap="coolwarm")
plt.title("Correlation between features")
plt.show()

Now training a simple logistic regression model to predict heart disease.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix

X = df.drop("target", axis=1)
y = df["target"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

pred = model.predict(X_test)
prob = model.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, pred))
print("ROC AUC:", roc_auc_score(y_test, prob))
print(confusion_matrix(y_test, pred))

## 3. Signal Data - ECG (MIT-BIH dataset)

For this part I used the wfdb library to fetch an actual ECG recording from PhysioNet (record 100).

In [ ]:
import wfdb

record = wfdb.rdrecord('100', pn_dir='mitdb')
ann = wfdb.rdann('100', 'atr', pn_dir='mitdb')

print(record.p_signal.shape)
print("sampling freq:", record.fs)
print("no. of beats:", len(ann.sample))

In [ ]:
# plotting first 5 sec of signal with the annotated beats
fs = record.fs
sig = record.p_signal[:fs*5, 0]
beats = ann.sample[ann.sample < fs*5]

plt.figure(figsize=(10,4))
plt.plot(sig)
plt.scatter(beats, sig[beats], color="red", label="beats")
plt.title("ECG signal - record 100")
plt.legend()
plt.show()

In [ ]:
# rough heart rate from RR intervals
rr = np.diff(ann.sample) / fs
print("avg RR interval:", rr.mean())
print("approx heart rate (bpm):", 60/rr.mean())

## 4. Textual Data

Taking a small sample clinical note and doing basic text processing on it (tokenizing + word frequency).

In [ ]:
note = """
Patient presents with intermittent chest pain radiating to the left arm,
associated with shortness of breath and mild sweating. Pain worsens with
exertion and improves with rest. No prior history of cardiac events.
Blood pressure elevated at time of visit. Recommend ECG and cardiac enzyme panel.
"""

import re
from collections import Counter

tokens = re.findall(r"[a-zA-Z]+", note.lower())

stop_words = {"with","to","the","and","at","of","no","prior","time","visit"}
clean_tokens = [t for t in tokens if t not in stop_words]

freq = Counter(clean_tokens)
freq.most_common(10)

## 5. Image Data

The lab mentions this is optional, so I used a synthetic grayscale image here instead of downloading an actual X-ray (avoids broken links in Colab).

In [ ]:
from PIL import Image

np.random.seed(1)
arr = (np.random.rand(256,256)*255).astype(np.uint8)
img = Image.fromarray(arr, mode="L")

resized = img.resize((128,128))

fig, ax = plt.subplots(1,2, figsize=(8,4))
ax[0].imshow(img, cmap="gray")
ax[0].set_title("original")
ax[1].imshow(resized, cmap="gray")
ax[1].set_title("resized")
plt.show()

## 6. Integration Discussion

The tabular model gives a heart disease risk based on patient stats, but it could be made better by adding features from the ECG signal, like heart rate variability from the RR intervals in part 3. Similarly the clinical note in part 4 has symptom words like "chest" and "pain" which could be turned into extra features and added to the same model. If all of this is linked to the same patient ID, it becomes one combined dataset that a model can be trained on instead of using each modality separately.